# Notebook 01: Train + Generate with EngiOpt CGAN-2D (DCC26)

Reference implementation for method integration, diagnostics, and artifact contract creation.


**Edit-safe start:** this notebook opens from GitHub in read-only source mode. Use **File -> Save a copy in Drive** before running edits so your changes stay in your own workspace.


## Notebook map

This notebook is written as a standalone lab chapter:
- context first,
- implementation second,
- interpretation third.

If you are following asynchronously, run cells in order and use the success checks to validate each stage before moving on.


## Standalone guide

This notebook is designed to be publication-grade reproducibility scaffolding:
clear controls, explicit artifacts, and interpretable training diagnostics.


In [ ]:
# Colab/local dependency bootstrap
import sys

IN_COLAB = 'google.colab' in sys.modules
FORCE_INSTALL = False  # Set True to force install outside Colab
ENGIOPT_GIT = 'git+https://github.com/IDEALLab/EngiOpt.git@codex/dcc26-workshop-notebooks#egg=engiopt'

if IN_COLAB or FORCE_INSTALL:
    print('Installing dependencies...')
    !pip install engibench[beams2d] sqlitedict torch torchvision matplotlib pandas tqdm tyro wandb
    !pip install {ENGIOPT_GIT}
    print('Dependency install complete.')
else:
    print('Skipping install (using current environment). Set FORCE_INSTALL=True to install here.')


## Part A: Configuration and runtime controls

Establish deterministic setup and artifact policy before touching model code.


### EngiBench vs EngiOpt roles in this notebook

EngiBench provides the benchmark contract; EngiOpt provides the method implementation.
Conflating these layers is a common source of irreproducible claims.


### Step 1 - Configure reproducible environment

Seed control and path control are first-class experimental settings, not boilerplate.


In [ ]:
import json
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch as th
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from engibench.problems.beams2d.v0 import Beams2D

try:
    from engiopt.cgan_2d.cgan_2d import Generator as EngiOptCGAN2DGenerator
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        'Could not import engiopt model class. Run the bootstrap cell first; on Colab, restart runtime after install if needed.'
    ) from exc

# Optional W&B integration
USE_WANDB_ARTIFACTS = False
WANDB_PROJECT = 'dcc26-workshop'
WANDB_ENTITY = None
WANDB_ARTIFACT_NAME = 'dcc26_beams2d_generated_artifacts'
WANDB_ARTIFACT_ALIAS = 'latest'
WANDB_LOG_TRAINING = True


def resolve_artifact_dir(create: bool = False) -> Path:
    in_colab = 'google.colab' in sys.modules
    path = Path('/content/dcc26_artifacts') if in_colab else Path('workshops/dcc26/artifacts')
    if create:
        path.mkdir(parents=True, exist_ok=True)
    return path


SEED = 7
random.seed(SEED)
np.random.seed(SEED)
th.manual_seed(SEED)
if th.cuda.is_available():
    th.cuda.manual_seed_all(SEED)

DEVICE = th.device('cuda' if th.cuda.is_available() else 'cpu')
print('device:', DEVICE)

ARTIFACT_DIR = resolve_artifact_dir(create=True)
print('artifact dir:', ARTIFACT_DIR)

CKPT_PATH = ARTIFACT_DIR / 'engiopt_cgan2d_generator_supervised.pt'
HISTORY_PATH = ARTIFACT_DIR / 'training_history.csv'
TRAIN_CURVE_PATH = ARTIFACT_DIR / 'training_curve.png'
LATENT_DIM = 32


## Part B: Load Beams2D data and build a stable workshop subset

We intentionally constrain runtime while preserving the full benchmark interaction pattern.


### Training diagnostics to monitor

Track trend shape (stability/collapse), not only endpoint value.
Diagnostics are evidence, not decoration.


### Step 2 - Build training slice from EngiBench dataset

Create aligned condition/design tensors and document scaling assumptions.


In [ ]:
problem = Beams2D(seed=SEED)
train_ds = problem.dataset['train']
test_ds = problem.dataset['test']

condition_keys = problem.conditions_keys
print('condition keys:', condition_keys)

N_TRAIN = 512
subset_idx = np.random.default_rng(SEED).choice(len(train_ds), size=N_TRAIN, replace=False)

conds_np = np.stack([np.array(train_ds[k])[subset_idx].astype(np.float32) for k in condition_keys], axis=1)
designs_np = np.array(train_ds['optimal_design'])[subset_idx].astype(np.float32)

# EngiOpt CGAN generator emits tanh-scaled outputs in [-1, 1].
targets_np = (designs_np * 2.0) - 1.0

print('conditions shape:', conds_np.shape)
print('designs shape:', designs_np.shape)
print('target range:', float(targets_np.min()), 'to', float(targets_np.max()))


### Step 3 - Define EngiOpt model and optimization objects

Model dimensions should derive from benchmark metadata, not hard-coded guesswork.


In [ ]:
model = EngiOptCGAN2DGenerator(
    latent_dim=LATENT_DIM,
    n_conds=conds_np.shape[1],
    design_shape=problem.design_space.shape,
).to(DEVICE)

optimizer = th.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()


def sample_noise(batch_size: int) -> th.Tensor:
    return th.randn((batch_size, LATENT_DIM), device=DEVICE, dtype=th.float32)


### Step 4 - Train (or load) with diagnostics

Persist checkpoint and training traces so downstream evaluation can be audited and repeated.


In [ ]:
TRAIN_FROM_SCRATCH = True
# Train on compact subset for workshop-time reliability (not full benchmark SOTA mode).
EPOCHS = 8
BATCH_SIZE = 64

train_losses = []
wandb_train_run = None

if TRAIN_FROM_SCRATCH:
    ds = TensorDataset(th.tensor(conds_np), th.tensor(targets_np))
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True)

    if USE_WANDB_ARTIFACTS and WANDB_LOG_TRAINING:
        import wandb

        wandb_train_run = wandb.init(
            project=WANDB_PROJECT,
            entity=WANDB_ENTITY,
            job_type='train',
            config={
                'seed': SEED,
                'epochs': EPOCHS,
                'batch_size': BATCH_SIZE,
                'n_train': int(N_TRAIN),
                'latent_dim': LATENT_DIM,
                'model_family': 'engiopt.cgan_2d.Generator',
            },
            reinit=True,
        )

    for epoch in range(EPOCHS):
        # Epoch-average loss is what we compare across quick workshop runs.
        model.train()
        epoch_loss = 0.0

        for cond_batch, target_batch in dl:
            cond_batch = cond_batch.to(DEVICE)
            target_batch = target_batch.to(DEVICE)

            pred = model(sample_noise(cond_batch.shape[0]), cond_batch)
            loss = criterion(pred, target_batch)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += float(loss.item())

        epoch_avg = epoch_loss / len(dl)
        train_losses.append(epoch_avg)
        print(f'epoch {epoch + 1:02d}/{EPOCHS} - loss: {epoch_avg:.4f}')

        if wandb_train_run is not None:
            wandb_train_run.log({'train/loss': epoch_avg, 'epoch': epoch + 1})

    th.save(
        {
            'model': model.state_dict(),
            'condition_keys': condition_keys,
            'latent_dim': LATENT_DIM,
            'model_family': 'engiopt.cgan_2d.Generator',
        },
        CKPT_PATH,
    )
    print('saved checkpoint to', CKPT_PATH)

    history_df = pd.DataFrame({'epoch': np.arange(1, len(train_losses) + 1), 'train_loss': train_losses})
    history_df.to_csv(HISTORY_PATH, index=False)

    fig, ax = plt.subplots(figsize=(6, 3.5))
    ax.plot(history_df['epoch'], history_df['train_loss'], marker='o')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE loss')
    ax.set_title('Notebook 01 training curve')
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(TRAIN_CURVE_PATH, dpi=150)
    plt.show()

    if wandb_train_run is not None:
        import wandb

        wandb_train_run.log({'train/loss_curve': wandb.Image(str(TRAIN_CURVE_PATH))})
        wandb_train_run.finish()
elif CKPT_PATH.exists():
    ckpt = th.load(CKPT_PATH, map_location=DEVICE)
    model.load_state_dict(ckpt['model'])
    model.eval()
    print('loaded checkpoint from', CKPT_PATH)

    if HISTORY_PATH.exists():
        history_df = pd.read_csv(HISTORY_PATH)
    else:
        history_df = pd.DataFrame(columns=['epoch', 'train_loss'])
else:
    raise FileNotFoundError(f'No checkpoint found at {CKPT_PATH}. Set TRAIN_FROM_SCRATCH=True or provide a checkpoint.')


## Part C: Condition-driven generation and quick sanity checks

Generate candidates for held-out conditions and run quick feasibility checks before full simulation.


### Scientific checkpoint before Notebook 02

Ask whether outputs are merely plausible-looking or genuinely evaluation-ready.
Notebook 02 will resolve this quantitatively.


### Step 5 - Generate conditioned designs

Use consistent test sampling so comparisons remain interpretable across runs.


In [ ]:
rng = np.random.default_rng(SEED)
N_SAMPLES = 24
selected = rng.choice(len(test_ds), size=N_SAMPLES, replace=False)

test_conds = np.stack([np.array(test_ds[k])[selected].astype(np.float32) for k in condition_keys], axis=1)
baseline_designs = np.array(test_ds['optimal_design'])[selected].astype(np.float32)

model.eval()
with th.no_grad():
    tanh_out = model(sample_noise(N_SAMPLES), th.tensor(test_conds, device=DEVICE))
    gen_designs_t = ((tanh_out.clamp(-1.0, 1.0) + 1.0) / 2.0).clamp(0.0, 1.0)
gen_designs = gen_designs_t.detach().cpu().numpy().astype(np.float32)

conditions_records = []
for i in range(N_SAMPLES):
    rec = {}
    for j, k in enumerate(condition_keys):
        v = test_conds[i, j]
        rec[k] = bool(v) if k == 'overhang_constraint' else float(v)
    conditions_records.append(rec)

# Quick checks before full Notebook 02 evaluation
viol_ratio = np.mean([
    len(problem.check_constraints(design=d, config=cfg)) > 0
    for d, cfg in zip(gen_designs, conditions_records, strict=True)
])
print('generated shape:', gen_designs.shape)
print('baseline shape:', baseline_designs.shape)
print('generated violation ratio (quick check):', float(viol_ratio))


### Step 6 - Export artifact contract

These files are the reproducible boundary between modeling and evaluation notebooks.


In [ ]:
# Artifact contract consumed by Notebook 02 and optional W&B logging.
generated_path = ARTIFACT_DIR / 'generated_designs.npy'
baseline_path = ARTIFACT_DIR / 'baseline_designs.npy'
conditions_path = ARTIFACT_DIR / 'conditions.json'

np.save(generated_path, gen_designs)
np.save(baseline_path, baseline_designs)
with open(conditions_path, 'w', encoding='utf-8') as f:
    json.dump(conditions_records, f, indent=2)

print('Saved artifacts to', ARTIFACT_DIR)
print('-', generated_path)
print('-', baseline_path)
print('-', conditions_path)
print('-', CKPT_PATH)
print('-', HISTORY_PATH)
print('-', TRAIN_CURVE_PATH)

if USE_WANDB_ARTIFACTS:
    try:
        import wandb

        run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, job_type='artifact-upload', reinit=True)
        run.log({
            'artifact/generated_mean_density': float(gen_designs.mean()),
            'artifact/generated_std_density': float(gen_designs.std()),
        })

        artifact = wandb.Artifact(
            WANDB_ARTIFACT_NAME,
            type='dataset',
            description='DCC26 Notebook 01 artifacts: arrays, conditions, checkpoint, and training diagnostics.',
        )
        for path in [generated_path, baseline_path, conditions_path, CKPT_PATH, HISTORY_PATH, TRAIN_CURVE_PATH]:
            if path.exists():
                artifact.add_file(str(path))
        run.log_artifact(artifact, aliases=[WANDB_ARTIFACT_ALIAS])
        run.finish()
        print('Uploaded artifacts to W&B:', WANDB_ARTIFACT_NAME)
    except Exception as exc:
        print('W&B upload failed (continuing with local artifacts only):', exc)


### Step 7 - Quick visual QA

Visual checks detect obvious failure patterns early (blank/noise/mode collapse).


In [ ]:
# Visual side-by-side snapshot (generated vs baseline)
fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for i in range(6):
    axes[0, i].imshow(gen_designs[i], cmap='gray', vmin=0, vmax=1)
    axes[0, i].set_title(f'gen {i}')
    axes[0, i].axis('off')

    axes[1, i].imshow(baseline_designs[i], cmap='gray', vmin=0, vmax=1)
    axes[1, i].set_title(f'base {i}')
    axes[1, i].axis('off')

fig.tight_layout()
plt.show()


## Troubleshooting

If a section fails, do not continue downstream. Fix locally first, then rerun the section and its immediate checks.
This notebook is intentionally staged so failures are localized.


## Next

Continue with Notebook 02 for benchmark-grade evaluation and reporting outputs.


## Takeaways

Before closing, record three points:
1. What conclusion is directly supported by your metrics?
2. What remains uncertain (and why)?
3. What extra experiment would you run next to reduce that uncertainty?
